<a href="https://colab.research.google.com/github/Squirrelcoding/green-space-vs-heat/blob/main/Green_Space_vs_Heat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
%pip install gdal

In [ ]:
!pip3 install OSMPythonTools
!pip3 install gdal

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 4.7 MB/s eta 0:00:00
  Created wheel for OSMPythonTools: filename=OSMPythonTools-0.3.6-py3-none-any.whl size=32932 sha256=6483d7bb69247c6c480fe301917a1e60399b8438881e24d607faa0bc8b345785
  Stored in directory: /root/.cache/pip/wheels/07/21/11/fe6b449f01c415e06e93a04360d8e03e3ac3908748e8143f5f
Successfully built OSMPythonTools


In [ ]:
import ee

# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library.
ee.Initialize(project='poopnet-4fb22')

In [ ]:
# TODO: Write some code that gets all of the boundaries of a parking lot given its ID or something
from OSMPythonTools.api import Api
api = Api()


In [ ]:
import geemap

In [ ]:
dataset = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2').filterDate(
    '2025-05-01', '2025-06-01'
)


# Applies scaling factors.
def apply_scale_factors(image):
  optical_bands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
  thermal_bands = image.select('ST_B.*').multiply(0.00341802).add(149.0)
  return image.addBands(optical_bands, None, True).addBands(
      thermal_bands, None, True
  )


dataset = dataset.map(apply_scale_factors)

visualization = {
    'bands': ['SR_B4', 'SR_B3', 'SR_B2'],
    'min': 0.0,
    'max': 0.3,
}

m = geemap.Map()
m.set_center(-114.2579, 38.9275, 8)
m.add_layer(dataset, visualization, 'True Color (432)')
m

Map(center=[38.9275, -114.2579], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=Search…

In [ ]:
naip_dataset = ee.ImageCollection('USDA/NAIP/DOQQ').filter(ee.Filter.date('2023-01-01', '2024-01-01'))

true_color = naip_dataset.select(['R', 'G', 'B'])

true_color_vis = {
  min: 0,
  max: 255,
};

m = geemap.Map()

m.setCenter(-73.9958, 40.7278, 15)
m.addLayer(true_color, true_color_vis, 'True Color')

m

Map(center=[40.7278, -73.9958], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchD…

In [ ]:
# Define ROI
west = -93.237314
south = 44.972722
east = -93.235855
north = 44.973557
roi = ee.Geometry.Rectangle([west, south, east, north])

# Mosaic and calculate NDVI
mosaic = naip_dataset.mosaic()
ndvi = mosaic.normalizedDifference(['N', 'R']).rename('NDVI')

# Scale for visualization
ndvi_vis = ndvi.multiply(127.5).add(127.5).uint8()

# Clip and export
clipped = ndvi_vis.clip(roi)

geemap.ee_export_image(
    clipped,
    filename='ndvi_vis.tif',
    scale=0.10394100844860077,
    region=roi
)


Generating URL ...
Please wait ...
Data downloaded to /content/ndvi_vis.tif


In [ ]:
import os
from osgeo import gdal
from PIL import Image

def tif_to_png(filename: str | os.PathLike):
  pre = filename.split(".")[0]
  print(pre)
  tif = gdal.Open(filename)
  arr = tif.ReadAsArray() # Convert CHW to HWC
  img = Image.fromarray(arr.astype(np.uint8))
  os.remove(filename)
  img.save(pre + '.png')

tif_to_png("ndvi_vis.tif")

ndvi_vis
